# Självstudieövningar – Vecka 6

De här övningarna är till för att du ska repetera och befästa det vi har gått igenom under veckan samt introducera några nya begrepp på ett tryggt sätt.

Det innebär att vissa av uppgifterna kan du behöva söka efter information själv hur du löser. Här kan du använda Google eller AI om du vill men dokumentationerna är din bästa vän när du lär dig.
- Python: https://docs.python.org/3.13/
- Matplotlib: https://matplotlib.org/stable/
- Pandas: https://pandas.pydata.org/
- NumPy: https://numpy.org/

Läs först igenom vad som ska göras i respektive uppgift och fyll sedan i kodblocket nedanför, eller gör uppgiften i en separat fil.

**Detta lämnas inte in för rättning.** Arbeta i din egen takt, testa dig fram, använd dokumentationen när du kör fast och ta med eventuella frågor till nästa lektionstillfälle.

## (Kör först) Skapa veckans dataset

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(123)

n = 800
cities = np.array(["Göteborg","Malmö","Stockholm","Uppsala"])
genders = np.array(["F","M"])
groups = np.array(["A","B"])  # för A/B-experiment

city = rng.choice(cities, size=n, p=[0.28, 0.22, 0.40, 0.10])
gender = rng.choice(genders, size=n)
age = rng.integers(18, 65, size=n)

# A/B-tilldelning (balanced med lite slump)
group = rng.choice(groups, size=n)

# Aktivitet (ungefär som "engagement score")
base = rng.normal(60, 12, size=n)

# Små effekter från stad/kön
city_shift = {"Göteborg":2, "Malmö":-1, "Stockholm":1, "Uppsala":0}
gender_shift = {"F":1.5, "M":0}
score = base + np.vectorize(city_shift.get)(city) + np.vectorize(gender_shift.get)(gender)

# Effekt av grupp: B ger +2 i snitt
score = score + np.where(group=="B", 2.0, 0.0)

# Begränsa och runda
score = np.clip(score, 10, 100)

# Binärt utfall (t.ex. "köp"): sannolikhet ökar med score
p_buy = 1/(1 + np.exp(-(score-55)/10))
purchase = rng.binomial(1, p_buy)

# Intäkter för köp (lognormal-ish), 0 om inget köp
revenue = np.where(purchase==1, rng.lognormal(mean=3.2, sigma=0.5, size=n), 0.0)

# Lägg in lite saknade värden
df = pd.DataFrame({
    "city": city,
    "gender": gender,
    "age": age,
    "group": group,
    "score": np.round(score,1),
    "purchase": purchase,
    "revenue": np.round(revenue,2)
})
mask = rng.random(size=df.shape) < 0.01  # ~1% NaN
for (r,c),m in np.ndenumerate(mask):
    if m:
        df.iat[r,c] = np.nan

df.to_csv("customer_activity.csv", index=False)
print("Skapade customer_activity.csv med", len(df), "rader.")

### Om datasetet (`customer_activity.csv`)

Det här är ett syntetiskt dataset från en fiktiv digital tjänst. Varje rad representerar en kundobservation.

| Kolumn | Typ | Beskrivning |
|---|---|---|
| `city` | kategori | Kundens stad |
| `gender` | kategori | Kön (F/M) |
| `age` | heltal | Ålder |
| `group` | kategori | Grupp A eller B |
| `score` | decimaltal | Aktivitet eller engagemang, 10–100 |
| `purchase` | 0/1 | Om kunden genomförde ett köp |
| `revenue` | decimaltal | Intäkt, 0 om inget köp genomfördes |

Grupp B har en liten positiv effekt på `score`. Sannolikheten för ett köp ökar med högre `score`. Datasetet innehåller även några saknade värden för att ge möjlighet att öva datarengöring.

Eftersom datasetet är syntetiskt ska det användas för att träna analysmetoder. Resultaten beskriver inte verkliga kunder.

### Snabb kontroll
- Läs in filen till `df` och förhandsgranska.
- Kontrollera datatyper och antal saknade värden per kolumn.
- Skriv en kort kommentar (i kod) om något du reagerar på (t.ex. att `revenue` ofta är 0 p.g.a. icke-köp).

## Block A – Sampling och variation

### Uppgift 1 – Slumpmässiga stickprov

Arbeta med kolumnen `score`.

- Ta bort saknade värden.
- Dra ett enkelt slumpmässigt stickprov med 100 observationer.
- Beräkna medelvärdet i stickprovet och jämför med medelvärdet i hela datasetet.
- Upprepa stickprovsdragningen 100 gånger och spara varje stickprovsmedelvärde.
- Rita ett histogram över de 100 stickprovsmedelvärdena.
- Beräkna medelvärde och standardavvikelse för stickprovsmedelvärdena.
- Kommentera kort varför stickprovsmedelvärdena varierar trots att de kommer från samma dataset.

Se till att stickprovsdragningen går att reproducera.

### Uppgift 2 – Bootstrap av medelvärdet

Skapa först en array utan saknade värden:

```python
score = df["score"].dropna().to_numpy()
```
Genomför därefter en bootstrapanalys:
- skapa 1 000 bootstrapstickprov genom att dra observationer från score med återläggning,
- låt varje bootstrapstickprov ha samma storlek som den ursprungliga arrayen,
- beräkna medelvärdet i varje bootstrapstickprov,
- visa bootstrapfördelningen i ett histogram,
- skriv ut det ursprungliga medelvärdet,
- skriv ut bootstrapmedelvärdenas medelvärde och standardavvikelse.

Använd en seed så att resultatet går att reproducera.  

*(Tips: En slumptalsgenerator och indexdragning med rng.integers() kan användas.)*

### Uppgift 3 – Förstå reproducerbar slump

Studera följande kod:

```python
import numpy as np

rng = np.random.default_rng(42)

a = rng.random(3)
b = rng.random(3)

print(a)
print(b)
```
Trots att en seed har angetts innehåller `a` och `b` olika tal.
- Förklara varför resultaten är olika.
- Skapa en ny generator med samma seed.
- Generera sex tal med den nya generatorn.
- Kontrollera att de första tre talen motsvarar `a` och att de sista tre motsvarar `b`.
- Förklara kort vad reproducerbarhet innebär i detta sammanhang.

**Viktigt**: En seed gör inte varje uttag identiskt. Den gör att samma serie av uttag kan återskapas.

## Block B – Konfidensintervall

### Uppgift 4 – Konfidensintervall för ett medelvärde

Skatta ett 95-procentigt konfidensintervall för medelvärdet av `score` i grupp A.

- filtrera fram grupp A,
- ta bort saknade värden i `score`,
- beräkna stickprovsstorlek, medelvärde och stickprovsstandardavvikelse,
- beräkna standardfelet,
- använd normalapproximationen med det kritiska värdet 1,96,
- beräkna intervallets undre och övre gräns.

Redovisa:

- punktestimatet,
- stickprovsstandardavvikelsen,
- standardfelet,
- konfidensintervallet.

Skriv även en kort tolkning av vad intervallet betyder.

Använd stickprovsstandardavvikelsen med `ddof=1`.

### Uppgift 5 – Konfidensintervall för en andel

Skatta ett 95-procentigt konfidensintervall för andelen kunder som genomfört ett köp i grupp B.

- filtrera fram grupp B,
- ta bort rader där `purchase` saknas,
- beräkna antalet observationer `n`,
- beräkna den observerade andelen köp,
- beräkna standardfelet för en andel,
- använd det kritiska värdet 1,96,
- beräkna intervallets undre och övre gräns.

Redovisa punktestimat, standardfel och konfidensintervall.

Kontrollera även om det förväntade antalet köp och icke-köp är tillräckligt stort för att normalapproximationen ska vara rimlig.

### Uppgift 6 – Felsökning: standardavvikelse och standardfel

Koden ska beräkna ett 95-procentigt konfidensintervall för medelvärdet av `score` i grupp A, men standardavvikelsen används på fel sätt.

Kör koden och rätta beräkningen:

```python
import numpy as np
import pandas as pd

df = pd.read_csv("customer_activity.csv")

x = df.loc[df["group"] == "A", "score"].dropna()

mean = x.mean()
sd = x.std()
n = x.shape[0]
z = 1.96

lo = mean - z * sd
hi = mean + z * sd

print(lo, hi)
```

Den rättade lösningen ska:

- beräkna standardfelet,
- använda standardfelet i konfidensintervallet,
- skriva ut medelvärde, standardavvikelse, standardfel och intervall,
- kort förklara skillnaden mellan standardavvikelse och standardfel.

In [ ]:
# Din rättade kod här

## Block C – Grundläggande hypotesprövning

### Uppgift 7 – A/B-test av medelvärden

Undersök om det genomsnittliga värdet på `score` skiljer sig mellan grupp A och grupp B.

- formulera nollhypotes och mothypotes,
- välj en signifikansnivå,
- ta bort saknade värden,
- beräkna antal observationer och medelvärde för respektive grupp,
- genomför ett tvåsidigt Welch t-test för oberoende grupper,
- rapportera teststatistika och p-värde,
- fatta ett beslut utifrån den valda signifikansnivån.

Redovisa även den observerade skillnaden mellan gruppernas medelvärden.

Avsluta med att skilja mellan:

- statistisk signifikans,
- skillnadens praktiska betydelse.

**Tips:** Funktionen `scipy.stats.ttest_ind()` kan användas med `equal_var=False`.

### Uppgift 8 – Felsökning: tolkning av p-värde

Följande kod drar en felaktig slutsats:

```python
p_value = 0.07

if p_value < 0.1:
    print("Signifikant på 5%-nivån!")
```

Rätta koden och resonemanget.

- Använd signifikansnivån 5 procent.
- Låt programmet skriva ut en korrekt slutsats.
- Förklara kort varför ett p-värde på `0.07` inte är statistiskt signifikant på 5-procentsnivån.
- Förklara varför detta inte innebär att nollhypotesen har bevisats vara sann.

In [ ]:
# Din rättade kod här